---
**Author:** Leonardo Gabriel Mourao Thiel  
**Project:** Master Thesis – System Inertia in the Energy System of the Future:
Model-Based Cost Optimization to Secure Inertia Requirements

**Notebook:**  Visualization and Analysis of Future System Inertia Scenarios (2040)


**Date:** 27.04.2026  
---

# Visualization and Analysis of Future System Inertia Scenarios (2040)

This notebook presents the visualization and analysis of system inertia
in a future European power system for the year 2040.

The results are based on an optimization model that simulates different
system configurations under varying assumptions regarding inertia provision.

---

## Background

The transition towards a low-carbon energy system is expected to
significantly change the generation mix, with a growing share of
renewable energy sources.

Since many renewable technologies do not inherently provide rotational
inertia, this transition may lead to reduced system inertia and increased
challenges for frequency stability.

To address this, alternative sources of inertia — such as virtual inertia
or advanced control strategies — are considered.

---

## Objective

The objective of this notebook is to analyze and compare system inertia
across different future scenarios:

- quantify inertia levels under different system configurations  
- identify critical low-inertia conditions  
- analyze temporal and spatial patterns  
- evaluate the impact of virtual inertia technologies  

---

## Scenarios

The following scenarios are considered:

- **No inertia**  
  Baseline case without explicit inertia constraints  

- **Thermal only**  
  Inertia provided exclusively by conventional generation  

- **Thermal + virtual**  
  Inclusion of virtual inertia sources  

These scenarios allow for a systematic comparison of different approaches
to maintaining system stability.

---

## Methodological Approach

The analysis builds on the following workflow:

1. Solve the optimization model for each scenario  
2. Extract system variables (generation, inertia, flows)  
3. Compute system inertia \( H_{sys}(t) \)  
4. Perform statistical and temporal analysis  
5. Visualize results across scenarios and countries  

---

## Key Metrics

The following indicators are used:

- **System inertia \( H_{sys}(t) \)**  
- **Share of low-inertia hours**  
- **Statistical measures (mean, median, variability)**  
- **Temporal profiles (hourly and seasonal patterns)**  

---

## Scope

- Year: 2040  
- Temporal resolution: hourly  
- Spatial scope: multi-country European system  
- Scenario-based analysis  

---

## Output

The notebook generates:

- comparative visualizations across scenarios  
- temporal profiles of system inertia  
- statistical summaries  
- indicators of critical system conditions  

These results provide insights into the effectiveness of different
inertia provision strategies in future power systems.

---

## Interpretation Focus

The analysis aims to answer:

- How does system inertia change across scenarios?  
- Which configurations lead to critical low-inertia conditions?  
- Can virtual inertia mitigate stability risks?  

---

## Notes

This notebook focuses on visualization and interpretation.
The optimization model and data preprocessing are implemented
in separate notebooks.

All results are structured to support reproducibility and
integration into the thesis.

## 1. Setup and Configuration

This section initializes the environment, imports required libraries,
and defines global parameters such as file paths, country selection,
and scenario configurations.

In [1]:
"""
Initial setup for data processing and analysis.

This section imports required libraries, defines file paths,
and initializes key parameters such as the list of countries,
solution file locations, and the analysis time horizon.
"""
# Ensure required packages are installed from requirements file
import sys
!{sys.executable} -m pip install -r requirements.txt
import pandas as pd
import sys
from inputs import InputConfig, InputLoader



# Import custom plotting module (assumed to contain visualization functions)
import Plots_2040 as p




# Base directory for input data
path = "../Data"

# Directory where results (e.g., plots, processed outputs) will be stored
output = "../Results/2040"


# List of countries included in the analysis (European region)
countryList = [
    "AL","AT","BA","BE","BG","CH","CZ","DE","DK","ES","FR","GR"
]

# Extend the list with additional countries
countryList += [
    "HR","HU","IT","LU","MK","ME","NL","PL","PT","RO","RS","SI","SK"
]


# Dictionary containing paths to optimization result files 
# Each key represents a scenario, and the value is the corresponding solution file

USE_FULL_RUN = True

if USE_FULL_RUN:
    base_path = "../Results/gurobi_out/2040_summer_01-07_to_31_08/"
else:
    base_path = "../Results/gurobi_out/"

sol_paths = {
    "no_inertia": base_path + "no_inertia.sol",
    "thermal_only": base_path + "thermal_only.sol",
    "thermal_plus_virtual": base_path + "thermal_plus_virtual.sol"
}

# Define the time horizon for analysis (summer period 2040)
start_date = "2040-07-01"
end_date   = "2040-09-01"

# Compute the duration of the analysis period in days
# Converts string dates to datetime objects before subtraction
duration = (pd.to_datetime(end_date) - pd.to_datetime(start_date)).days

ERROR: Could not find a version that satisfies the requirement os (from versions: none)

[notice] A new release of pip is available: 25.3 -> 26.1
[notice] To update, run: C:\Users\Leo\AppData\Local\Python\pythoncore-3.12-64\python.exe -m pip install --upgrade pip
ERROR: No matching distribution found for os


## 2. Data Loading

In this section, the input data is loaded using a configuration-based approach.
The `InputConfig` object defines the temporal scope, data location, and
selected countries. The `InputLoader` then processes this configuration
to retrieve the required datasets.

In [2]:

# Create a configuration object that defines how data should be loaded
# This includes:
# - path: base directory of the input data
# - start_date / end_date: temporal filtering of the dataset
# - countryList: subset of countries to include in the analysis
cfg = InputConfig(
    path,
    start_date,
    end_date,
    countryList
)

# Load input data based on the configuration
# The loader is expected to:
# - read data files from disk
# - filter by date range and countries
# - return structured data (e.g., DataFrames or custom objects)
inputs = InputLoader.load(cfg)

## 3. Solution Loading and Parsing

This section loads optimization results from different scenarios and
transforms them into a structured format suitable for analysis.

Each solution file is parsed to extract relevant variables and filtered
based on the selected countries.

In [3]:



# ---------------------------------------------------------
# 1) LOAD AND PROCESS SOLUTION FILES
# ---------------------------------------------------------

# Dictionary to store processed solution data for each scenario
solutions = {}

# Iterate over all defined scenarios and their corresponding file paths
for name, path in sol_paths.items():

    # Read raw solution file (e.g., Gurobi .sol format)
    # Expected to return a DataFrame containing variable names and values
    df = p.read_sol_file(path)

    # Parse and structure variables from the raw solution
    # Likely extracts meaningful components (e.g., country, timestep, variable type)
    # and filters the data based on the selected country list
    df = p.parse_variables(df, countryList)

    # Store processed DataFrame under the scenario name
    solutions[name] = df


## 4. Scenario-Based Analysis Pipeline

This section implements the main analysis pipeline. For each scenario,
a sequence of computations and visualizations is executed, including:

- generation analysis
- system inertia evaluation (H_sys)
- statistical analysis of critical periods
- visualization of dispatch and energy mix

The pipeline ensures consistent processing across all scenarios,
enabling meaningful comparisons.

In [ ]:
# =========================================================
# MASTER PIPELINE: GENERATE ALL PLOTS (plots_2040)
# =========================================================

import os

# Display available scenarios for verification
print("Loaded scenarios:", list(solutions.keys()))


# ---------------------------------------------------------
# 2) MAIN LOOP: PROCESS EACH SCENARIO
# ---------------------------------------------------------

for scenario, df in solutions.items():

    print(f"\n--- Processing scenario: {scenario} ---")

    # -----------------------------------------------------
    # Generation analysis
    # -----------------------------------------------------

    # Compute detailed generation split by fuel type
    detailed = p.compute_generation_split_by_fuel(df, inputs)

    # Visualize generation composition
    p.plot_generation_pie_detailed(detailed, scenario)

    # Plot dispatch behavior for the scenario
    p.plot_dispatch_by_scenario(df, inputs, scenario)


    # =====================================================
    # H_sys + statistical analysis
    # =====================================================

    # Extract system inertia (H_sys) per country
    H_df = p.extract_H_sys_per_country(df, inputs, countryList)

    if not H_df.empty:

        # Compute statistical metrics (mean, median, std, etc.)
        stats = p.compute_stats(H_df)

        # Identify critical periods with high demand and low inertia
        df_merged = p.compute_critical_high_demand(H_df, inputs, countryList)

        # Visualizations of critical system conditions
        p.plot_critical_high_demand(df_merged, scenario)
        p.plot_mean_median_std(stats, scenario)
        p.plot_critical_hours(stats, scenario)

        # Heatmap and temporal analysis of inertia
        p.plot_H_sys_heatmap(H_df, scenario)
        p.plot_Hsys_vs_demand(H_df, inputs, scenario)
        p.plot_critical_hours_by_time_and_country(H_df, scenario)



    # =====================================================
    # Generation shares
    # =====================================================

    # Compute relative generation shares per country
    shares = p.compute_generation_shares(df, inputs, countryList)

    # Plot generation shares
    p.plot_generation_shares(shares, scenario)


    # =====================================================
    # Inertia constraints / slack
    # =====================================================

    # Analyze slack variables related to inertia constraints
    p.plot_inertia_slack(df, scenario)


    # =====================================================
    # Virtual inertia analysis
    # =====================================================

    # Compute percentage contribution of virtual inertia
    df_perc = p.compute_virtual_percentages(df, countryList)

    if not df_perc.empty:
        # Save results for further analysis (e.g., thesis appendix)
        df_perc.to_excel(output + "/inertia/virtual_percentages.xlsx", index=False)

    # Compute inertia share per country
    inertia_share = p.compute_inertia_share_per_country(df, countryList)

    if not inertia_share.empty:
        p.plot_inertia_share_per_country(inertia_share, scenario)

    # Detailed breakdown of virtual inertia sources
    df_virtual = p.compute_virtual_inertia_breakdown(df, countryList)

    if not df_virtual.empty:
        p.plot_virtual_inertia_breakdown(df_virtual)


# ---------------------------------------------------------
# 3) CROSS-SCENARIO ANALYSIS
# ---------------------------------------------------------

print("\n--- Cross-scenario analysis ---")

# Compare overall scenario performance
df = p.compare_scenarios(solutions, sol_paths, inputs, countryList)

# Compare generation splits across scenarios
df_compare = p.compare_generation_split(solutions, inputs)

# Export comparison results to Excel
with pd.ExcelWriter(output + "/compare_scenarios.xlsx") as writer:
    df.to_excel(writer, sheet_name="Overview", index=False)
    df_compare.to_excel(writer, sheet_name="Generation_Split")


# ---------------------------------------------------------
# System-wide comparative plots
# ---------------------------------------------------------

p.plot_dispatch_daily_avg_all(solutions, inputs)
p.plot_battery_charge_all_countries(solutions, countryList)

# Generation mix comparison
df_mix = p.compute_generation_mix_per_country_scenarios(solutions, inputs, countryList)

for c in countryList:
    p.plot_generation_mix_per_country(df_mix, c)

p.plot_generation_mix_all_countries(df_mix, countryList)

# Battery usage comparison
p.plot_battery_by_scenario(solutions)

# Generation change analysis
delta_gen = p.compare_generation_shares(solutions, inputs, countryList)
p.plot_generation_change(delta_gen)


# ---------------------------------------------------------
# Curtailment analysis
# ---------------------------------------------------------

p.plot_curtailment(solutions, countryList)

delta_curt = p.compare_curtailment_vs_base(solutions, inputs, countryList)
p.plot_curtailment_change(delta_curt)


# ---------------------------------------------------------
# Network / flow analysis
# ---------------------------------------------------------

# Net imports
p.plot_net_imports(solutions, countryList)

# Total flows
flow_results = p.compare_total_flows(solutions)
p.plot_total_flows(flow_results)

# Flow changes relative to baseline
flow_delta = p.compare_total_flows_with_base(solutions)
p.plot_flow_changes(flow_delta)
p.plot_flow_percent(flow_delta)

# Identify most significant flow changes
p.plot_top_flow_changes(solutions)


# ---------------------------------------------------------
# Additional analyses
# ---------------------------------------------------------

# Change in system inertia
p.plot_delta_Hsys(solutions, inputs, countryList)

# Demand-side response (DSR) utilization
df_dsr = p.compute_dsr_utilization(solutions, inputs, countryList)
p.plot_dsr_utilization_per_scenario(df_dsr)

# Correlation analysis (NOTE: uses last computed H_df!)
p.plot_Hsys_correlation(H_df)

# Export generation shares
p.export_generation_shares_all_scenarios(solutions, inputs, countryList)
# Batterynetto profiles
p.plot_battery_net_per_country(solutions,countryList)



print("All plots generated and saved")

Loaded scenarios: ['no_inertia', 'thermal_only', 'thermal_plus_virtual']

--- Processing scenario: no_inertia ---
Saved: ../Results/2040\generation\no_inertia_generation_pie.pdf
Saved: ../Results/2040\generation\no_inertia_generation_pie.png
